In [1]:
# ── Cell 1 (002_bpe): The problem with char-level, made visible ──
text = "the theme of the theatre is there"

# Char-level: every character is a token
char_tokens = list(text)
print("Char-level tokens:", len(char_tokens))
print(char_tokens)

# Notice: 'the' appears 4 times but char-level re-spells it every time:
# t-h-e ... t-h-e ... t-h-e ... t-h-e — hugely wasteful.
# BPE will learn 'the' as ONE token.

Char-level tokens: 33
['t', 'h', 'e', ' ', 't', 'h', 'e', 'm', 'e', ' ', 'o', 'f', ' ', 't', 'h', 'e', ' ', 't', 'h', 'e', 'a', 't', 'r', 'e', ' ', 'i', 's', ' ', 't', 'h', 'e', 'r', 'e']


In [2]:
# ── Cell 2: Count how often each adjacent pair appears ───────────
def get_pair_counts(tokens):
    """Count every adjacent pair in the token list.
    tokens = ['t','h','e',' ','t','h','e'] ->
    pairs: ('t','h'):2, ('h','e'):2, ('e',' '):1, (' ','t'):1"""
    counts = {}
    for pair in zip(tokens, tokens[1:]):   # slide a 2-wide window
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# Try it on our example
tokens = list("the theme of the theatre is there")
counts = get_pair_counts(tokens)

# Show the most frequent pairs first
for pair, c in sorted(counts.items(), key=lambda x: -x[1])[:6]:
    print(f"{pair} -> {c}")

('t', 'h') -> 5
('h', 'e') -> 5
('e', ' ') -> 4
(' ', 't') -> 4
('r', 'e') -> 2
('e', 'm') -> 1


In [3]:
# ── Cell 3: Merge every occurrence of a chosen pair ──────────────
def merge(tokens, pair, new_token):
    """Replace every adjacent occurrence of `pair` with `new_token`.
    tokens=['t','h','e'], pair=('t','h'), new_token='th'
        -> ['th','e']"""
    new_tokens = []
    i = 0
    while i < len(tokens):
        # If this position + next match the pair, merge them
        if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
            new_tokens.append(new_token)
            i += 2                    # skip both — they became one
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

# Demo: one full BPE step on our example
tokens = list("the theme of the theatre is there")
print("Before:", len(tokens), "tokens")

counts = get_pair_counts(tokens)
top_pair = max(counts, key=counts.get)          # most frequent pair
print("Merging top pair:", top_pair)

merged = "".join(top_pair)                        # ('t','h') -> 'th'
tokens = merge(tokens, top_pair, merged)
print("After: ", len(tokens), "tokens")
print(tokens)

Before: 33 tokens
Merging top pair: ('t', 'h')
After:  28 tokens
['th', 'e', ' ', 'th', 'e', 'm', 'e', ' ', 'o', 'f', ' ', 'th', 'e', ' ', 'th', 'e', 'a', 't', 'r', 'e', ' ', 'i', 's', ' ', 'th', 'e', 'r', 'e']


In [4]:
# ── Cell 4: Run many merges = train the tokenizer ────────────────
def train_bpe(text, num_merges):
    """Learn `num_merges` merge rules from text.
    Returns the final tokens + the ordered list of merges learned."""
    tokens = list(text)          # start: every char is a token
    merges = []                  # record merges IN ORDER (order matters!)

    for step in range(num_merges):
        counts = get_pair_counts(tokens)
        if not counts:
            break
        top_pair = max(counts, key=counts.get)
        if counts[top_pair] < 2:      # nothing repeats enough — stop early
            break
        new_token = "".join(top_pair)
        tokens = merge(tokens, top_pair, new_token)
        merges.append((top_pair, new_token))
        print(f"merge {step+1:2d}: {top_pair} -> '{new_token}'  ({counts[top_pair]}x)")

    return tokens, merges

# Train on our toy sentence
text = "the theme of the theatre is there the theatre theme"
final_tokens, merges = train_bpe(text, num_merges=10)

print("\nStarted with", len(list(text)), "char tokens")
print("Ended with  ", len(final_tokens), "tokens:", final_tokens)
print("\nLearned vocabulary of merges:", [m[1] for m in merges])

merge  1: ('t', 'h') -> 'th'  (8x)
merge  2: ('th', 'e') -> 'the'  (8x)
merge  3: (' ', 'the') -> ' the'  (7x)
merge  4: ('r', 'e') -> 're'  (3x)
merge  5: (' the', 'm') -> ' them'  (2x)
merge  6: (' them', 'e') -> ' theme'  (2x)
merge  7: (' the', ' the') -> ' the the'  (2x)
merge  8: (' the the', 'a') -> ' the thea'  (2x)
merge  9: (' the thea', 't') -> ' the theat'  (2x)
merge 10: (' the theat', 're') -> ' the theatre'  (2x)

Started with 51 char tokens
Ended with   13 tokens: ['the', ' theme', ' ', 'o', 'f', ' the theatre', ' ', 'i', 's', ' the', 're', ' the theatre', ' theme']

Learned vocabulary of merges: ['th', 'the', ' the', 're', ' them', ' theme', ' the the', ' the thea', ' the theat', ' the theatre']


In [5]:
# ── Cell 5: Turn text <-> integer IDs using learned merges ───────
def encode_bpe(text, merges):
    """Encode new text by REPLAYING merges in the same order learned."""
    tokens = list(text)                     # start as chars
    for pair, new_token in merges:          # apply each merge, in order
        tokens = merge(tokens, pair, new_token)
    return tokens

def build_vocab(merges, base_chars):
    """Assign an integer ID to every token (base chars + merged tokens)."""
    vocab = {ch: i for i, ch in enumerate(sorted(base_chars))}
    for _, new_token in merges:
        if new_token not in vocab:
            vocab[new_token] = len(vocab)
    inv_vocab = {i: t for t, i in vocab.items()}
    return vocab, inv_vocab

# ── Put it together ──
text = "the theme of the theatre is there the theatre theme"
final_tokens, merges = train_bpe(text, num_merges=10)
base_chars = set(text)

vocab, inv_vocab = build_vocab(merges, base_chars)
print("Vocab size:", len(vocab))

# Encode a NEW sentence using the learned merges
sample = "the theatre"
str_tokens = encode_bpe(sample, merges)          # -> ['the', ' ', 'theatre'] etc.
ids = [vocab[t] for t in str_tokens]             # -> integer IDs for the model
print("\nText:       ", sample)
print("BPE tokens: ", str_tokens)
print("Token IDs:  ", ids)

# Decode back
decoded = "".join(inv_vocab[i] for i in ids)
print("Decoded:    ", decoded)
print("Round-trip OK:", decoded == sample)

merge  1: ('t', 'h') -> 'th'  (8x)
merge  2: ('th', 'e') -> 'the'  (8x)
merge  3: (' ', 'the') -> ' the'  (7x)
merge  4: ('r', 'e') -> 're'  (3x)
merge  5: (' the', 'm') -> ' them'  (2x)
merge  6: (' them', 'e') -> ' theme'  (2x)
merge  7: (' the', ' the') -> ' the the'  (2x)
merge  8: (' the the', 'a') -> ' the thea'  (2x)
merge  9: (' the thea', 't') -> ' the theat'  (2x)
merge 10: (' the theat', 're') -> ' the theatre'  (2x)
Vocab size: 21

Text:        the theatre
BPE tokens:  ['the', ' the', 'a', 't', 're']
Token IDs:   [12, 13, 1, 10, 14]
Decoded:     the theatre
Round-trip OK: True


In [6]:
# ── Cell 6: A self-contained BPE tokenizer for Yuti ──────────────
class BPETokenizer:
    """Trains on a corpus, then encodes/decodes text <-> token IDs.
    Same interface as our char tokenizer (encode/decode/vocab_size),
    so Yuti's model code needs ZERO changes."""

    def __init__(self):
        self.merges = []
        self.vocab = {}
        self.inv_vocab = {}

    def train(self, text, num_merges):
        tokens = list(text)
        base_chars = set(text)
        self.merges = []

        for _ in range(num_merges):
            counts = get_pair_counts(tokens)
            if not counts:
                break
            top = max(counts, key=counts.get)
            if counts[top] < 2:
                break
            new_tok = "".join(top)
            tokens = merge(tokens, top, new_tok)
            self.merges.append((top, new_tok))

        # Build integer vocab: base chars first, then merged tokens
        self.vocab = {ch: i for i, ch in enumerate(sorted(base_chars))}
        for _, new_tok in self.merges:
            if new_tok not in self.vocab:
                self.vocab[new_tok] = len(self.vocab)
        self.inv_vocab = {i: t for t, i in self.vocab.items()}

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text):
        tokens = list(text)
        for pair, new_tok in self.merges:
            tokens = merge(tokens, pair, new_tok)
        return [self.vocab[t] for t in tokens]

    def decode(self, ids):
        return "".join(self.inv_vocab[i] for i in ids)


# ── Test on real data ──
with open("input.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

tok = BPETokenizer()
tok.train(corpus, num_merges=500)     # learn 500 merge rules


print("Vocab size:", tok.vocab_size)  # ~500+ base chars

sample = corpus[:200]
ids = tok.encode(sample)
print("\nOriginal chars:", len(sample))
print("BPE tokens:    ", len(ids))
print("Compression:   ", round(len(sample) / len(ids), 2), "chars per token")
print("\nRound-trip OK: ", tok.decode(ids) == sample)

Vocab size: 576

Original chars: 200
BPE tokens:     145
Compression:    1.38 chars per token

Round-trip OK:  True


In [7]:
# ── Cell 7: BPE corpus -> tensors -> train/val split ─────────────
import torch

# Encode the ENTIRE corpus with our trained BPE tokenizer
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
print("Total BPE tokens:", len(data), "(was", len(corpus), "chars)")

n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print("Train:", len(train_data), "| Val:", len(val_data))

# Config for v2 (note: vocab_size now comes from BPE, not char count)
cfg = {
    "batch_size": 32, "block_size": 64,
    "n_embd": 128, "n_head": 4, "n_layer": 4, "dropout": 0.1,
    "learning_rate": 3e-4, "max_iters": 1500, # 3000,
    "eval_interval": 300, "eval_iters": 200,
    "vocab_size": tok.vocab_size,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
print("vocab_size:", cfg["vocab_size"])

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - cfg["block_size"], (cfg["batch_size"],))
    x = torch.stack([d[i:i+cfg["block_size"]] for i in ix])
    y = torch.stack([d[i+1:i+cfg["block_size"]+1] for i in ix])
    return x.to(cfg["device"]), y.to(cfg["device"])

xb, yb = get_batch("train")
print("Batch shapes:", xb.shape, yb.shape)

Total BPE tokens: 56685 (was 144696 chars)
Train: 51016 | Val: 5669
vocab_size: 576
Batch shapes: torch.Size([32, 64]) torch.Size([32, 64])


In [8]:
import torch
from torch import nn
from torch.nn import functional as F

# ── Cell 10 (v2): Building blocks — Head, MultiHead, FF, Block ────

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(cfg["n_embd"], head_size, bias=False)
        self.query = nn.Linear(cfg["n_embd"], head_size, bias=False)
        self.value = nn.Linear(cfg["n_embd"], head_size, bias=False)
        self.register_buffer("tril",
            torch.tril(torch.ones(cfg["block_size"], cfg["block_size"])))
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        _B, T, _C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, cfg["n_embd"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(cfg["dropout"]),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

print("Building blocks defined ✓")

Building blocks defined ✓


In [9]:
# ── Cell 11 (fixed): YutiGPT takes vocab_size as a parameter ─────
class YutiGPT(nn.Module):
    def __init__(self, vocab_size):                    # ← accept it
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, cfg["n_embd"])
        self.position_embedding_table = nn.Embedding(cfg["block_size"], cfg["n_embd"])
        self.blocks = nn.Sequential(
            *[Block(cfg["n_embd"], cfg["n_head"]) for _ in range(cfg["n_layer"])]
        )
        self.ln_f = nn.LayerNorm(cfg["n_embd"])
        self.lm_head = nn.Linear(cfg["n_embd"], vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -cfg["block_size"]:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = YutiGPT(cfg["vocab_size"]).to(cfg["device"])   # ← pass it here
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
model

Parameters: 948,032


YutiGPT(
  (token_embedding_table): Embedding(576, 128)
  (position_embedding_table): Embedding(64, 128)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=128, out_features=32, bias=False)
            (query): Linear(in_features=128, out_features=32, bias=False)
            (value): Linear(in_features=128, out_features=32, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias

In [10]:
# ── Cell 8: Train Yuti on BPE tokens ─────────────────────────────
torch.manual_seed(1337)
model = YutiGPT(cfg["vocab_size"]).to(cfg["device"])
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"])

@torch.no_grad()
def estimate_loss():
    out = {}; model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(cfg["eval_iters"])
        for k in range(cfg["eval_iters"]):
            X, Y = get_batch(split); _, l = model(X, Y); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train(); return out

for it in range(cfg["max_iters"]):
    if it % cfg["eval_interval"] == 0 or it == cfg["max_iters"]-1:
        l = estimate_loss()
        print(f"step {it:4d} | train {l['train']:.4f} | val {l['val']:.4f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()

Parameters: 948,032
step    0 | train 6.5282 | val 6.5226
step  300 | train 4.6293 | val 4.8754
step  600 | train 3.9573 | val 4.3619
step  900 | train 3.6460 | val 4.2222
step 1200 | train 3.3907 | val 4.1574
step 1499 | train 3.1093 | val 4.0970


In [11]:
ctx = torch.zeros((1,1), dtype=torch.long, device=cfg["device"])
print(tok.decode(model.generate(ctx, max_new_tokens=200)[0].tolist()))


and same sea—ooptseep to,” saids tak— the ner—and d aI can c,” said Alice. “Wellsaid to lekepulat!

    The EnThe Mock Turtle March Hare Sool Alice, so Seal,
    I shalf an exty you, White Caterpentrened sivem question a
Her.




CQate she had not!”

“TER Ctainnee
Butes with
Tute to it: and little Pion him what’s you She said Turtle.
“Cuppoor not this more,” said the Cat, “I must becamedons you getting upon a pieest. “Well s
silent’s ‘Come reay you! Ohs you ked the Gryphon.

“Oh, _I_
